# GPU Stock Screener, Colab Runner

Runs the [gpu-stock-screener](https://github.com/arhaanb/gpu-stock-screener) project end to end on a Colab GPU runtime.

**Before running:** go to `Runtime` > `Change runtime type` and pick a GPU (T4 is fine). The screener also works on a CPU runtime but the backtest is slower.

**Cells:**
1. Clone or pull the repo.
2. Verify the GPU is visible to CuPy.
3. Install dependencies.
4. Clean old outputs.
5. Quick run (single strategy, no backtest).
6. Full run (every strategy with the rolling backtest).
7. Inspect leaderboards, plots, the animated GIF, and metrics.

## 1. Clone or pull the repo

Conditional: if the folder already exists from a previous run, we pull the latest changes. Otherwise we clone fresh.

In [ ]:
import os

REPO_URL = "https://github.com/arhaanb/gpu-stock-screener.git"
REPO_DIR = "gpu-stock-screener"

if os.path.exists(REPO_DIR):
    print(f"{REPO_DIR} already exists, pulling latest")
    !cd {REPO_DIR} && git pull
else:
    print(f"cloning {REPO_URL}")
    !git clone {REPO_URL}

%cd {REPO_DIR}

## 2. Verify the GPU

`nvidia-smi` shows the card Colab assigned, and importing CuPy confirms it can talk to the driver.

In [ ]:
!nvidia-smi || echo 'no nvidia-smi, running on cpu'

In [ ]:
try:
    import cupy as cp
    print("cupy version:", cp.__version__)
    n = cp.cuda.runtime.getDeviceCount()
    print("visible devices:", n)
    if n > 0:
        print("device 0:", cp.cuda.runtime.getDeviceProperties(0)['name'].decode())
except Exception as e:
    print("cupy not available, will fall back to numpy:", e)

## 3. Install dependencies

Colab already has CuPy preinstalled for CUDA 12, so we only need numpy, pandas, yfinance, matplotlib, and pillow.

In [ ]:
!pip install -q -r requirements.txt

## 4. Clean old outputs

Wipes everything in `output/` and `execution_artefacts/` so the next run is fresh. The `data/cache/` price cache is left alone so we do not re-download Yahoo prices every time. Set `HARD_RESET = True` below to also drop the cache.

In [ ]:
HARD_RESET = False

if HARD_RESET:
    !make reset
else:
    !make clean

## 5. Quick run, single strategy

Sanity check: run the Sharpe strategy on the default universe with no backtest. This produces a leaderboard, a risk vs return scatter, a sector heatmap, and a price grid of the top 20 picks.

In [ ]:
!python -m src.screener \
    --tickers data/tickers/sp100.txt \
    --strategy sharpe \
    --lookback 60 \
    --top 20 \
    --start 2019-01-01 \
    --end 2024-12-31 \
    --verbose

## 6. Full run, every strategy with rolling backtest

This is the main event. For every strategy we run the full five year rolling backtest, rebalancing every 21 trading days, and compare to SPY buy and hold. Each strategy writes its own leaderboard CSV, equity curve CSV, metrics JSON, scatter, sector heatmap, equity curve PNG, and animated leaderboard GIF.

In [ ]:
!python -m src.screener \
    --tickers data/tickers/sp100.txt \
    --strategy all \
    --lookback 60 \
    --top 20 \
    --start 2019-01-01 \
    --end 2024-12-31 \
    --backtest \
    --rebalance 21 \
    --verbose

## 7. Inspect the outputs

### Leaderboards

In [ ]:
import pandas as pd
from pathlib import Path

for csv_path in sorted(Path('output').glob('leaderboard_*.csv')):
    print(f"\n=== {csv_path.stem} ===")
    df = pd.read_csv(csv_path)
    display(df.head(10))

### Backtest metrics

In [ ]:
import json
from pathlib import Path

rows = []
for js in sorted(Path('output').glob('metrics_*.json')):
    strategy = js.stem.replace('metrics_', '')
    with open(js) as fh:
        metrics = json.load(fh)
    metrics = {'strategy': strategy, **metrics}
    rows.append(metrics)

if rows:
    import pandas as pd
    summary = pd.DataFrame(rows).set_index('strategy')
    display(summary.style.format({
        'total_return': '{:.2%}',
        'benchmark_total_return': '{:.2%}',
        'cagr': '{:.2%}',
        'benchmark_cagr': '{:.2%}',
        'sharpe': '{:.2f}',
        'max_drawdown': '{:.2%}',
        'benchmark_max_drawdown': '{:.2%}',
    }))
else:
    print('no metrics json yet, did the backtest run?')

### Static plots

All the PNGs from `execution_artefacts/`. Scatter, sector heatmap, price grid, equity curve for each strategy.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for png in sorted(Path('execution_artefacts').glob('*.png')):
    print(png.name)
    display(Image(str(png)))

### Animated leaderboard GIFs

One GIF per strategy showing how the top 10 picks shift over the backtest window.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for gif in sorted(Path('execution_artefacts').glob('*.gif')):
    print(gif.name)
    display(Image(str(gif)))